# ETL Silver → Gold

Popula o Star Schema no PostgreSQL a partir dos dados limpos do Silver Layer.

In [309]:
# Verificacao rapida do estado das tabelas
import psycopg2

DB_CONFIG_CHECK = {
    'host': 'localhost',
    'port': 5432,
    'database': 'amazon_sales',
    'user': 'postgres',
    'password': 'postgres'
}

conn_check = psycopg2.connect(**DB_CONFIG_CHECK)
cur_check = conn_check.cursor()

print("="*60)
print("DIAGNOSTICO PRE-ETL")
print("="*60)

# Silver
cur_check.execute("SELECT COUNT(*) FROM silver.product")
silver_total = cur_check.fetchone()[0]
print(f"\nSILVER:")
print(f"  Total produtos: {silver_total:,}")

cur_check.execute("SELECT COUNT(*) FROM silver.product WHERE purchased_last_month IS NOT NULL AND purchased_last_month > 0")
silver_com_vendas = cur_check.fetchone()[0]
print(f"  Com vendas: {silver_com_vendas:,}")

# Gold
cur_check.execute("SELECT COUNT(*) FROM gold.dim_prdt")
gold_prdt = cur_check.fetchone()[0]
cur_check.execute("SELECT COUNT(*) FROM gold.dim_tmp")
gold_tmp = cur_check.fetchone()[0]
cur_check.execute("SELECT COUNT(*) FROM gold.dim_cat")
gold_cat = cur_check.fetchone()[0]
cur_check.execute("SELECT COUNT(*) FROM gold.ft_vnd")
gold_vnd = cur_check.fetchone()[0]

print(f"\nGOLD (antes do ETL):")
print(f"  dim_prdt: {gold_prdt:,}")
print(f"  dim_tmp: {gold_tmp:,}")
print(f"  dim_cat: {gold_cat:,}")
print(f"  ft_vnd: {gold_vnd:,}")

print("\n" + "="*60)

if silver_com_vendas == 0:
    print("\n*** ALERTA: Nao ha produtos com vendas na silver! ***")
    print("O ETL populara as dimensoes, mas a tabela fato ficara vazia.")

cur_check.close()
conn_check.close()


DIAGNOSTICO PRE-ETL

SILVER:
  Total produtos: 15,938
  Com vendas: 11,585

GOLD (antes do ETL):
  dim_prdt: 8,378
  dim_tmp: 6
  dim_cat: 22
  ft_vnd: 11,585



## 1. Imports

Bibliotecas necessárias para o ETL.


In [310]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch
import warnings

warnings.filterwarnings('ignore', message='.*SQLAlchemy.*')

## 2. Configuracao

Parametros de conexao ao banco de dados.


In [311]:
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'amazon_sales',
    'user': 'postgres',
    'password': 'postgres'
}

## 3. Carregar Dados do Silver

Leitura dos dados da tabela `silver.product` no PostgreSQL.


In [312]:
print("Conectando ao PostgreSQL...")
conn = psycopg2.connect(**DB_CONFIG)
print("Conectado")

# Verificar se a tabela tem dados
cur_check = conn.cursor()
cur_check.execute("SELECT COUNT(*) FROM silver.product")
total_rows = cur_check.fetchone()[0]
cur_check.close()
print(f"\nTotal de registros em silver.product: {total_rows:,}")

if total_rows == 0:
    print("Tabela silver.product vazia")

query = """
    SELECT 
        id,
        asin,
        title,
        brand,
        category,
        rating,
        total_reviews as review_count,
        purchased_last_month as units_sold_last_month,
        discounted_price as final_price,
        original_price,
        discount_percentage as discount_pct,
        is_best_seller as best_seller_badge,
        is_sponsored as sponsored_badge,
        has_coupon as has_active_coupon,
        buy_box_availability as available_for_purchase,
        quality_score,
        price_range as price_tier,
        date,
        time,
        EXTRACT(HOUR FROM time)::int as hour,
        -- Calcular is_promotable com base nos criterios:
        -- rating >= 4.0 AND total_reviews >= 100 AND purchased_last_month >= 200 AND buy_box_availability = true
        CASE 
            WHEN rating >= 4.0 
                AND total_reviews >= 100 
                AND purchased_last_month >= 200 
                AND buy_box_availability = true 
            THEN true 
            ELSE false 
        END as is_promotable,
        -- Calcular revenue_last_month
        (purchased_last_month * discounted_price) as revenue_last_month,
        -- Criar collected_at combinando date + time
        (date + time) as collected_at,
        -- price_imputation_tier (valor padrao)
        'original' as price_imputation_tier
    FROM silver.product
"""

df = pd.read_sql_query(query, conn)

Conectando ao PostgreSQL...
Conectado

Total de registros em silver.product: 15,938


## 4. Preparar Cursor

Cria cursor para operacoes no banco.


In [313]:
# Criar cursor para operacoes
cur = conn.cursor()
print("Cursor criado")

Cursor criado


## 5. Limpar Tabelas

Remove dados antigos para reprocessamento.


In [314]:
cur.execute("TRUNCATE TABLE gold.ft_vnd, gold.dim_prdt, gold.dim_tmp, gold.dim_cat CASCADE;")
conn.commit()
print("Tabelas limpas")

Tabelas limpas


## 6. Dimensão Tempo (dim_tmp)

Popula a dimensão temporal com atributos calculados.


In [315]:
dates = df['date'].dropna().unique()
data = []

# Para cada data única, calcular atributos temporais
for date_str in dates:
    dt = pd.to_datetime(date_str)
    data.append((
        dt.date(),                          # Data
        dt.year,                            # Ano
        dt.month,                           # Mês
        dt.day,                             # Dia
        dt.dayofweek,                       # Dia da semana (0=segunda)
        dt.day_name(),                      # Nome do dia
        (dt.month - 1) // 3 + 1,           # Trimestre
        dt.isocalendar()[1],               # Semana do ano
        dt.dayofweek >= 5,                 # É fim de semana?
        dt.strftime('%Y-%m'),              # Mês-ano
        f"{dt.year}-Q{(dt.month-1)//3 + 1}" # Ano-trimestre
    ))

execute_batch(cur, """
    INSERT INTO gold.dim_tmp 
    (data, ano, mes, dia, dia_semana, nome_dia_semana, trimestre, 
     semana_ano, eh_fim_semana, mes_ano, ano_trimestre)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (data) DO NOTHING
""", data)
conn.commit()

## 7. Dimensão Categoria (dim_cat)

Popula a dimensão de categorias com segmentos de negócio.


In [316]:
categorias = df['category'].dropna().unique()
data = []

# Mapeamento de categorias para segmentos de negócio
segmento_map = {
    'Audio': 'Eletrônicos', 'Camera': 'Eletrônicos', 'Mobile': 'Eletrônicos',
    'Laptop': 'Computadores', 'Storage': 'Computadores', 'Networking': 'Computadores',
    'Printing': 'Escritório', 'Power': 'Acessórios', 'Accessory': 'Acessórios'
}

# Construir registros
for cat in categorias:
    cat_str = str(cat) if pd.notna(cat) else 'Other'
    segmento = segmento_map.get(cat_str, 'Outros')
    data.append((cat_str, cat_str, segmento))

execute_batch(cur, """
    INSERT INTO gold.dim_cat (categoria, tipo_produto, segmento)
    VALUES (%s, %s, %s)
    ON CONFLICT (categoria) DO NOTHING
""", data)
conn.commit()

## 8. Dimensão Produto (dim_prdt)

Popula a dimensão de produtos com todos os ASINs únicos.


In [317]:
produtos = df[['asin', 'title', 'brand', 'category', 'price_tier',
               'best_seller_badge', 'sponsored_badge', 'is_promotable',
               'available_for_purchase']].drop_duplicates('asin')

data = []

# Construir tuplas com dados dos produtos
for _, row in produtos.iterrows():
    data.append((
        str(row['asin']),
        str(row['title'])[:500] if pd.notna(row['title']) else 'Unknown',
        str(row['brand']) if pd.notna(row['brand']) else 'Unknown',
        str(row['category']) if pd.notna(row['category']) else 'Other',
        str(row['price_tier']) if pd.notna(row['price_tier']) else 'Unknown',
        bool(row['best_seller_badge']) if pd.notna(row['best_seller_badge']) else False,
        bool(row['sponsored_badge']) if pd.notna(row['sponsored_badge']) else False,
        bool(row['is_promotable']) if pd.notna(row['is_promotable']) else False,
        bool(row['available_for_purchase']) if pd.notna(row['available_for_purchase']) else False
    ))

execute_batch(cur, """
    INSERT INTO gold.dim_prdt 
    (asin, titulo, marca, categoria, faixa_preco, best_seller_badge,
     sponsored_badge, is_promotable, disponivel_compra)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (asin) DO UPDATE SET
        titulo = EXCLUDED.titulo,
        best_seller_badge = EXCLUDED.best_seller_badge,
        is_promotable = EXCLUDED.is_promotable,
        data_atualizacao = NOW()
""", data, page_size=1000)
conn.commit()

## 9. Tabela Fato (ft_vnd)

Popula a tabela fato com vendas, conectando as 3 dimensões via FKs.


In [318]:
df_facts = df[df['units_sold_last_month'].notna()].copy()

# Buscar FKs das dimensões
cur.execute("SELECT asin, prdt_key FROM gold.dim_prdt")
asin_to_key = dict(cur.fetchall())

cur.execute("SELECT tmp_key, data FROM gold.dim_tmp")
date_to_key = {pd.to_datetime(d).date(): k for k, d in cur.fetchall()}

cur.execute("SELECT categoria, cat_key FROM gold.dim_cat")
cat_to_key = dict(cur.fetchall())

data = []
skipped = 0

# Construir registros da fato
for _, row in df_facts.iterrows():
    # Buscar FKs
    prdt_key = asin_to_key.get(str(row['asin']))
    tmp_key = date_to_key.get(pd.to_datetime(row['date']).date())
    cat_key = cat_to_key.get(str(row['category']) if pd.notna(row['category']) else 'Other')
    
    # Se alguma FK faltar, pular registro
    if not (prdt_key and tmp_key and cat_key):
        skipped += 1
        continue
    
    # Montar tupla com FKs + measures
    data.append((
        prdt_key, tmp_key, cat_key,
        int(row['units_sold_last_month']),
        float(row['revenue_last_month']) if pd.notna(row['revenue_last_month']) else 0.0,
        float(row['final_price']) if pd.notna(row['final_price']) else 0.0,
        float(row['rating']) if pd.notna(row['rating']) else None,
        int(row['review_count']) if pd.notna(row['review_count']) else 0,
        float(row['quality_score']) if pd.notna(row['quality_score']) else None,
        float(row['discount_pct']) if pd.notna(row['discount_pct']) else 0.0,
        int(row['hour']) if pd.notna(row['hour']) else 0,
        pd.to_datetime(row['collected_at']) if pd.notna(row['collected_at']) else None,
        str(row['price_imputation_tier']) if pd.notna(row['price_imputation_tier']) else 'unknown'
    ))

# Inserir em lote (FIX: usar prdt_srk, tmp_srk, cat_srk)
execute_batch(cur, """
    INSERT INTO gold.ft_vnd 
    (prdt_srk, tmp_srk, cat_srk, unidades_vendidas, receita_estimada,
     preco_final, rating, total_reviews, quality_score, percentual_desconto,
     hora_coleta, data_coleta, origem_preco)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
""", data, page_size=1000)
conn.commit()

## 10. Validação

Verifica contagens e calcula métricas de negócio.


In [319]:


# Contar registros em cada tabela
cur.execute("SELECT COUNT(*) FROM gold.dim_prdt")
cur.execute("SELECT COUNT(*) FROM gold.dim_tmp")
cur.execute("SELECT COUNT(*) FROM gold.dim_cat")
cur.execute("SELECT COUNT(*) FROM gold.ft_vnd")
vnd_count = cur.fetchone()[0]

if vnd_count > 0:
    print("\n Gold pronta!")
else:
    print("\n Fato vazia")

conn.close()


 Gold pronta!
